<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [24]:

date_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

print(date_check.to_string(index=False))


 row_count first_date  last_date
   9841378 2026-03-01 2026-03-31


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [26]:

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

field_check = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,
        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS missing_ga4_sessions
    FROM read_parquet('{march_path}')
""").df()

print(field_check.to_string(index=False))


 row_count  gsc_available_rows  ga4_available_rows  missing_gsc_clicks  missing_ga4_sessions
   9841378             3611061              413966                   0               3018741


- **Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`. These are observed performance signals, provided the relevant data is available.
- **Label:** Future performance outcome; not yet defined or verified.
- **Context:** `report_date`, `month`, `client_hash_id`, `content_hash_id`, `gsc_data_available`, `ga4_data_available`.
- **Excluded:** Future-period performance from features, because it would not be known at the decision moment and could cause data leakage.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

availability_check = con.execute(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_clicks IS NOT NULL
        ) AS usable_gsc_click_rows
    FROM read_parquet('{march_path}')
""").df()

print(availability_check.to_string(index=False))



 gsc_available_rows  ga4_available_rows  usable_gsc_click_rows
            3611061              413966                3611061


In [29]:

feature_frame = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    LIMIT 10
""").df()

print(feature_frame.to_string(index=False))


report_date          client_hash_id          content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  ga4_pageviews  ga4_sessions
 2026-03-01 client_73cda7b4e4f265ea content_b7e512995f79d5a6               20           0          3.350000           <NA>          <NA>
 2026-03-01 client_73cda7b4e4f265ea content_05597932fe4da067                1           0          0.000000           <NA>          <NA>
 2026-03-01 client_73cda7b4e4f265ea content_7a105f548d9c6916              125           1          4.928000           <NA>          <NA>
 2026-03-01 client_73cda7b4e4f265ea content_905aa32a0230694e                7           0          4.000000           <NA>          <NA>
 2026-03-01 client_73cda7b4e4f265ea content_a3ea9792f793ec72               11           0          2.272727           <NA>          <NA>
 2026-03-01 client_73cda7b4e4f265ea content_36c36abc7650d7af              239           1          7.347280           <NA>          <NA>
 2026-03-01 client_73cda7b4e4f265ea conte

**Five features and when they are available**

1. `gsc_impressions` — Knowable at the decision moment because it records search impressions observed up to that date, when GSC data is available.
2. `gsc_clicks` — Knowable at the decision moment because it records search clicks observed up to that date, when GSC data is available.
3. `gsc_avg_position` — Knowable at the decision moment because it summarizes observed search position, when GSC data is available.
4. `ga4_pageviews` — Knowable at the decision moment because it records pageviews observed up to that date, when GA4 data is available.
5. `ga4_sessions` — Knowable at the decision moment because it records sessions observed up to that date, when GA4 data is available.

**Note:** The sample rows show GA4 values as missing. Missing values must not be treated as zero.

In [30]:

leakage_check = con.execute("""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS march_clicks
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-03/data_0.parquet'
        )
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-04/data_0.parquet'
        )
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    joined AS (
        SELECT
            march.march_clicks,
            COALESCE(april.april_clicks, 0) AS future_clicks,
            CASE WHEN COALESCE(april.april_clicks, 0) > 0
                 THEN 1 ELSE 0 END AS label
        FROM march
        INNER JOIN april USING (client_hash_id, content_hash_id)
    )
    SELECT
        COUNT(*) AS rows_checked,
        AVG(label) AS future_positive_rate,
        AVG(CASE WHEN march_clicks > 0 THEN 1.0 ELSE 0.0 END)
            AS current_clicks_positive_rate
    FROM joined
""").df()

print(leakage_check.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rows_checked  future_positive_rate  current_clicks_positive_rate
       158549              0.373519                      0.429142


In [31]:

score_check = con.execute("""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS march_clicks
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-03/data_0.parquet'
        )
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-04/data_0.parquet'
        )
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    joined AS (
        SELECT
            march.march_clicks,
            CASE WHEN COALESCE(april.april_clicks, 0) > 0
                 THEN 1 ELSE 0 END AS label
        FROM march
        INNER JOIN april USING (client_hash_id, content_hash_id)
    )
    SELECT
        AVG(CASE WHEN (march_clicks > 0) =
                      (label = 1) THEN 1.0 ELSE 0.0 END)
            AS honest_baseline_accuracy,
        AVG(CASE WHEN label = label THEN 1.0 ELSE 0.0 END)
            AS leaked_accuracy
    FROM joined
""").df()

print(score_check.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 honest_baseline_accuracy  leaked_accuracy
                 0.798189              1.0


**Leakage experiment**

The future outcome is whether a content item received any GSC clicks in April 2026. The honest baseline accuracy was 79.82%. When the label itself was used as a feature, the score became 100%, which is perfect only because the answer was given to the model. This leaked feature must not be used for real prediction. The honest baseline is the comparison to retain; the perfect score is shown only to demonstrate leakage.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** March-to-April comparison includes only client-content pairs with GSC availability in both months, so the results may not represent all content items or clients. Observed clicks do not prove that a content refresh causes improvement.

In [32]:

limits_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS missing_ga4_sessions
    FROM read_parquet('{march_path}')
""").df()

print(limits_check.to_string(index=False))


 total_rows  gsc_available_rows  ga4_available_rows  missing_ga4_sessions
    9841378             3611061              413966               3018741


- This data shows observed associations, not proof that refreshing content causes more clicks.
- Client histories are unbalanced, so not every client has the same amount of historical data.
- GSC and GA4 availability differs; missing GA4 values must not be treated as zero.
- The March 2026 slice alone cannot establish future performance.
- Overlapping time windows may introduce leakage if future information is included in features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.